In [10]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캡을 열 수 업슷니다.")
    exit()
while True:
    ret, frame = cap.read()
    if not ret:
        print("프레임을 가져 올수 없습니다.")
        break
    flip_fram =cv2.flip(frame,1)
    height,width,_ = frame.shape
    center_x,center_y = width//2,height//2
    roi = flip_fram[center_y -150 :center_y+150,center_x-150:center_x+150]
    cv2.rectangle(flip_fram,(center_x - 150,center_y - 150),(center_x + 150,center_y + 150),(0,255,0),2)
    cv2.imshow('Webcam',flip_fram)
    #화면 캡쳐를 위한 키 값 받기-----------------------------------------
    key = cv2.waitKey(1) & 0xFF
    if key == ord('c'or 'C'): #c capture의 약자
        gray_img = cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)
        gray_img=np.flip(gray_img,1)
        cv2.imwrite('../ML/gray_image.png', gray_img)
        gaussian_blur = cv2.GaussianBlur(gray_img,(5,5),3)

        _,otsu_thread = cv2.threshold(gaussian_blur,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        cv2.imshow('otsu_thread',otsu_thread)

        #### Morph-----------------------------------------------
        kernel =np.ones((5,5),np.uint8)
        erosion=cv2.erode(otsu_thread,kernel,iterations = 5) #침식 반대 다일라이트?
        cv2.imshow('erosion',erosion)
        cv2.imwrite('../ML/digit_binary.png', erosion)

        #이미지 자르기-------------------------------------------------
        img = cv2.imread('../ML/digit_binary.png', cv2.IMREAD_UNCHANGED)
        h,w = img.shape[:2] #2개 가져와라
        crop_size =280 #가로 세로 이미지를 자름
        cx,cy = int(w/2),int(h/2)
        half = crop_size // 2
        x1,x2 =cx - half, cx + half
        y1,y2 = cy - half, cx + half

        #경계면 설정-----------------------------------------------
        x1= max(0,x1)
        y1= max(0,y1)
        x2= min(w,x2)
        y2= min(h,y2)

        cropped_img = img[y1:y2,x1:x2]
        cv2.imshow('cropped_img',cropped_img)

        #이미지 반전---------------------------------------------
        reversed_img = cv2.bitwise_not(cropped_img)
        cv2.imshow('reversed_img',reversed_img)
        cv2.imwrite('../ML/.IMAG_FOR_TEST.png', reversed_img)

        #28*28이라 축소 해줘야한다


    if cv2.waitKey(1) ==27:
        break
cap.release()
cv2.destroyAllWindows()
